# Story Director — Claude API

Generates viral YouTube Shorts scene manifests using Claude + your training data as few-shot context.

**How it works:**
1. Loads best creator wisdom from `training_data_v8.jsonl` (John Scott, Jenny Hoyos, Discord Q&A, synthetic scripts)
2. Injects it all into the Claude system prompt
3. Caches it with Anthropic prompt caching — first call writes cache, all calls after use it at 90% cheaper
4. You just provide a story prompt → get back a JSON scene manifest

**Cost:** ~\$0.05 first call (cache write), ~\$0.003 each call after

In [ ]:
# Cell 1 — Install
!pip install anthropic -q

In [ ]:
# Cell 2 — API Key
import os

# Option A: Colab secrets (recommended)
# Add your key in the left panel under the key icon, name it ANTHROPIC_API_KEY
try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
    print('Key loaded from Colab secrets')
except Exception:
    # Option B: paste directly
    os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-YOUR_KEY_HERE'
    print('Using hardcoded key')

In [ ]:
# Cell 3 — Download training data
import gdown, os

# Paste your file ID from the Google Drive share link:
# https://drive.google.com/file/d/FILE_ID_HERE/view
GDRIVE_FILE_ID = 'YOUR_FILE_ID_HERE'
DATASET_PATH   = '/content/training_data_v8.jsonl'

if not os.path.exists(DATASET_PATH):
    print('Downloading training_data_v8.jsonl...')
    gdown.download(f'https://drive.google.com/uc?id={GDRIVE_FILE_ID}', DATASET_PATH, quiet=False)
    print(f'Downloaded to {DATASET_PATH}')
else:
    print(f'Already exists: {DATASET_PATH}')

In [ ]:
# Cell 4 — Build system prompt from training data
import json
from collections import defaultdict
from pathlib import Path

DATASET_PATH = '/content/training_data_v8.jsonl'
MAX_TRAINING_EXAMPLES = 50
MIN_LEN = 150
MAX_LEN = 800

PRIORITY_SOURCES = {
    'john_scott_lesson',
    'jenny_hoyos_interview',
    'synthetic_script_30s',
    'synthetic_structure',
    'hook_formulas',
    'discord_qa',
    'yt_transcript_masterclass_creator_rant',
}
SOURCE_CAPS = {
    'john_scott_lesson': 7,
    'jenny_hoyos_interview': 10,
    'synthetic_script_30s': 8,
    'synthetic_structure': 8,
    'hook_formulas': 5,
    'discord_qa': 10,
    'yt_transcript_masterclass_creator_rant': 5,
}

SYSTEM_CORE = """You are a Story Director for viral short-form pixel-art animated videos in the style of PixelBeef (@pixelbeefshorts).

Your ONLY job: take a story prompt and output a structured JSON scene manifest.

## Beat Structure (mandatory)
1. hook        - visual + verbal punch in first 1.5-3 seconds. Promise something is coming.
2. foreshadow  - exactly 2 lines, under 3 seconds. Plant the seed.
3. obstacle    - the conflict hits. Use 'but' - something goes wrong.
4. amplifier   - stakes escalate. Use 'therefore' - things get worse.
5. twist       - unexpected reversal. Subvert what the viewer predicted.
6. payoff      - instant resolution. One expression. Cut immediately.

## Hard Rules
- Total runtime: 28-34 seconds (target 30s or 50s, never 20s or 40s)
- 5th-grade reading level, short punchy narration only
- But/Therefore storytelling, NEVER 'and then'
- End IMMEDIATELY after payoff, no breathing room
- emotion_intensity escalates: hook ~0.6 to payoff ~1.0
- Hook must drop in mid-action, no setup

## Output Format
Respond with ONLY valid JSON. No markdown. No explanation.

{\n  \"title\": \"string\",\n  \"target_duration_sec\": 30,\n  \"emotion\": \"suspense|comedy|horror|heartwarming|action|mystery|triumph|tragedy|shock|curiosity\",\n  \"scenes\": [\n    {\n      \"scene_id\": 1,\n      \"beat_type\": \"hook|foreshadow|obstacle|amplifier|twist|payoff\",\n      \"duration_sec\": 3,\n      \"narration\": \"spoken voiceover line\",\n      \"dialogue\": null,\n      \"characters_visible\": [\"character_name\"],\n      \"camera\": \"wide|medium|close|extreme_close|overhead|low_angle\",\n      \"visual_description\": \"what the viewer sees\",\n      \"action\": \"what happens physically in this beat\",\n      \"emotion_intensity\": 0.7\n    }\n  ]\n}"""

def load_training_wisdom(path):
    if not Path(path).exists():
        print(f'Dataset not found at {path}, running without training context')
        return ''

    by_source = defaultdict(list)
    with open(path, encoding='utf-8') as f:
        for line in f:
            try:
                rec = json.loads(line)
            except Exception:
                continue
            source = rec.get('source', '')
            if source not in PRIORITY_SOURCES:
                continue
            msgs     = rec.get('messages', [])
            user_msg = next((m['content'] for m in msgs if m['role'] == 'user'), '')
            asst_msg = next((m['content'] for m in msgs if m['role'] == 'assistant'), '')
            if len(asst_msg) < MIN_LEN:
                continue
            by_source[source].append({
                'weight': rec.get('weight', 3),
                'user': user_msg[:200],
                'assistant': asst_msg[:MAX_LEN],
            })

    pool = []
    for source, items in by_source.items():
        items.sort(key=lambda x: -x['weight'])
        cap = SOURCE_CAPS.get(source, 5)
        pool.extend(items[:cap])
    pool.sort(key=lambda x: -x['weight'])
    pool = pool[:MAX_TRAINING_EXAMPLES]

    if not pool:
        return ''

    lines = ['\n\n## Creator Wisdom (from real masterclasses and creator Q&A)\n',
             'Study these. They define what makes a Short go viral.\n']
    for ex in pool:
        lines.append(f"Q: {ex['user']}")
        lines.append(f"A: {ex['assistant']}\n")
    return '\n'.join(lines)

SYSTEM_PROMPT = SYSTEM_CORE + load_training_wisdom(DATASET_PATH)
print(f'System prompt built: ~{len(SYSTEM_PROMPT)//4:,} tokens')
print('First call writes the cache. All calls after use it at 90% cheaper.')

In [ ]:
# Cell 5 — Few-shot example
FEW_SHOT_PROMPT = 'A bus robbery where the woman had a decoy phone'

FEW_SHOT_MANIFEST = {
    'title': 'Not Her First Robbery',
    'target_duration_sec': 30,
    'emotion': 'suspense',
    'scenes': [
        {
            'scene_id': 1, 'beat_type': 'hook', 'duration_sec': 3,
            'narration': 'This bus is about to get robbed. But this woman has a plan.',
            'dialogue': None, 'characters_visible': ['woman', 'thief'],
            'camera': 'wide',
            'visual_description': 'City bus interior. Masked thief stands at front, gun raised.',
            'action': 'Thief raises gun, passengers freeze. Woman stays eerily calm.',
            'emotion_intensity': 0.7
        },
        {
            'scene_id': 2, 'beat_type': 'foreshadow', 'duration_sec': 3,
            'narration': 'One by one, he took from every passenger. She was next.',
            'dialogue': None, 'characters_visible': ['thief', 'woman'],
            'camera': 'medium',
            'visual_description': 'Thief moves down aisle collecting phones. Woman watches.',
            'action': 'Thief approaches row by row, woman hand moves toward purse',
            'emotion_intensity': 0.6
        },
        {
            'scene_id': 3, 'beat_type': 'obstacle', 'duration_sec': 5,
            'narration': 'But the moment his back turned, she started digging.',
            'dialogue': None, 'characters_visible': ['woman'],
            'camera': 'close',
            'visual_description': 'Woman hands frantically searching inside large purse.',
            'action': 'Woman pulls out phone, slides it under her leg',
            'emotion_intensity': 0.8
        },
        {
            'scene_id': 4, 'beat_type': 'amplifier', 'duration_sec': 5,
            'narration': 'Therefore when he got to her, she had a phone ready. Just not THE phone.',
            'dialogue': None, 'characters_visible': ['woman', 'thief'],
            'camera': 'close',
            'visual_description': 'Thief looms over her, pillowcase open. Woman reaches into purse, not under leg.',
            'action': 'Woman slowly produces a different phone, hands it over',
            'emotion_intensity': 0.9
        },
        {
            'scene_id': 5, 'beat_type': 'twist', 'duration_sec': 4,
            'narration': 'He took it. Walked off the bus. Never looked back.',
            'dialogue': None, 'characters_visible': ['woman'],
            'camera': 'medium',
            'visual_description': 'Thief exits. Door closes. Woman stares forward.',
            'action': 'Woman waits for door to close, then lifts her leg',
            'emotion_intensity': 0.95
        },
        {
            'scene_id': 6, 'beat_type': 'payoff', 'duration_sec': 3,
            'narration': 'Turns out this was not her first robbery.',
            'dialogue': None, 'characters_visible': ['woman'],
            'camera': 'extreme_close',
            'visual_description': 'Real phone in her hand, screen lit. Small smirk.',
            'action': 'Cut to black immediately',
            'emotion_intensity': 1.0
        }
    ]
}

print('Few-shot example ready (bus robbery decoy phone)')

In [ ]:
# Cell 6 — Generate your story
import anthropic

# ══════════════════════════════════════════════════════════════════════════
#  CHANGE YOUR STORY IDEA HERE  ↓↓↓
# ══════════════════════════════════════════════════════════════════════════
STORY_PROMPT = 'A cashier who gives exact change to the guy about to rob the store'
# ══════════════════════════════════════════════════════════════════════════


def generate_manifest(story_prompt, verbose=True):
    client = anthropic.Anthropic()

    response = client.messages.create(
        model='claude-opus-4-7',
        max_tokens=2048,
        system=[
            {
                'type': 'text',
                'text': SYSTEM_PROMPT,
                'cache_control': {'type': 'ephemeral'},  # cache all training wisdom
            }
        ],
        messages=[
            # Few-shot example also cached
            {
                'role': 'user',
                'content': [
                    {'type': 'text', 'text': FEW_SHOT_PROMPT,
                     'cache_control': {'type': 'ephemeral'}}
                ],
            },
            {
                'role': 'assistant',
                'content': json.dumps(FEW_SHOT_MANIFEST, indent=2),
            },
            # Your actual prompt
            {'role': 'user', 'content': story_prompt},
        ],
    )

    raw = response.content[0].text.strip()
    if raw.startswith('```'):
        raw = raw.split('```')[1]
        if raw.startswith('json'):
            raw = raw[4:]
        raw = raw.strip()

    manifest = json.loads(raw)

    if verbose:
        u = response.usage
        cache_read  = getattr(u, 'cache_read_input_tokens', 0)
        cache_write = getattr(u, 'cache_creation_input_tokens', 0)
        print(f'Tokens  input:{u.input_tokens}  output:{u.output_tokens}')
        if cache_write:
            print(f'  cache WRITE: {cache_write:,} tokens (paid once, free next time)')
        if cache_read:
            print(f'  cache READ : {cache_read:,} tokens (90% cheaper)')

    return manifest


def print_manifest(m):
    print(f"\n{'='*60}")
    print(f"  {m.get('title', 'Untitled')}")
    print(f"  {m.get('target_duration_sec', '?')}s  |  {m.get('emotion', '?')}")
    print(f"{'='*60}")
    for s in m.get('scenes', []):
        print(f"\n[{s['beat_type'].upper()}] {s['duration_sec']}s  cam:{s['camera']}  intensity:{s['emotion_intensity']}")
        print(f"  NARRATION: {s['narration']}")
        print(f"  ACTION:    {s['action']}")
    print(f"{'='*60}")


manifest = generate_manifest(STORY_PROMPT)
print_manifest(manifest)

print('\nFull JSON:')
print(json.dumps(manifest, indent=2))


## Try more prompts

Just change `STORY_PROMPT` in Cell 6 and re-run it.

The training context is cached after the first call — every subsequent call is ~90% cheaper.

**Prompt ideas:**
- `'A cashier who gives exact change to a customer who was going to rob the store'`
- `'A guy who finds a wallet and returns it, only to discover whose wallet it was'`
- `'A soldier who gets lost and finds an enemy soldier in the same situation'`
- `'A kid who discovers his imaginary friend was real all along'`